Ahora el dataset cambia de perspectiva. Antes: película → conjunto de usuarios. Ahora: usuario → conjunto de películas que le gustaron (basket). Buscamos películas que aparecen juntas frecuentemente.


FP-Growth: algoritmo eficiente basado en árbol, implementado en MLlib.
SON: Apriori distribuido en 2 pasadas sobre el RDD.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Fase4_Asociacion")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "4g")
    .config("spark.memory.fraction", "0.8")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.port", "4042")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo ✓")

Spark 4.1.1 listo ✓


# Datos reducidos

In [ ]:
# Celda 1 — top 20 películas, 5% usuarios
from pyspark.sql.functions import col, collect_list, count

ratings = spark.read.parquet("../outputs/results/ratings_clean.parquet")

# Solo top 20 películas
top_peliculas = (ratings
    .filter(col("like") == 1)
    .groupBy("movieId")
    .agg(count("*").alias("n_likes"))
    .orderBy("n_likes", ascending=False)
    .limit(20)
    .select("movieId")
)
ids_top = set(row["movieId"] for row in top_peliculas.collect())
print(f"Películas: {ids_top}")

# 5% de usuarios completos
usuarios_muestra = (ratings
    .select("userId").distinct()
    .sample(fraction=0.05, seed=42)
)

transacciones = (ratings
    .join(usuarios_muestra, on="userId", how="inner")
    .filter(col("like") == 1)
    .filter(col("movieId").isin(list(ids_top)))
    .groupBy("userId")
    .agg(collect_list("movieId").alias("items"))
    .filter("size(items) >= 2")
)
transacciones.cache()
print(f"Usuarios: {transacciones.count():,}")

ERROR:root:KeyboardInterrupt while sending command.][Stage 126:>  (0 + 0) / 1]
Traceback (most recent call last):
  File "/Users/johar/Desktop/Data_Mining/proyecto/proyecto_dm/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/johar/Desktop/Data_Mining/proyecto/proyecto_dm/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

# FP-Growth con MLlib

In [ ]:
# Solo esta celda — sin ningún .count()
from pyspark.ml.fpm import FPGrowth
import time

MIN_SUPPORT    = 0.20
MIN_CONFIDENCE = 0.5

t0 = time.time()
modelo_fp = FPGrowth(
    itemsCol="items",
    minSupport=MIN_SUPPORT,
    minConfidence=MIN_CONFIDENCE
).fit(transacciones)
t_fp = time.time() - t0
print(f"FP-Growth entrenado en: {t_fp:.1f}s")

# .show() en vez de .count() — solo materializa 15 filas
print("\nItemsets frecuentes (top 15):")
modelo_fp.freqItemsets.orderBy("freq", ascending=False).show(15)

print("\nReglas de asociación (top 15):")
modelo_fp.associationRules.orderBy("confidence", ascending=False).show(15, truncate=False)

FP-Growth: 2.7s


ERROR:root:KeyboardInterrupt while sending command.>                (0 + 2) / 8]
Traceback (most recent call last):
  File "/Users/johar/Desktop/Data_Mining/proyecto/proyecto_dm/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/johar/Desktop/Data_Mining/proyecto/proyecto_dm/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

# SON Algorithm (2 pasadas)

In [ ]:
import itertools
import time

# ── Parámetros ────────────────────────────────────────────────────────────────
MIN_SUPPORT_ABS  = int(MIN_SUPPORT * n_usuarios)   # soporte absoluto
num_particiones  = transacciones.rdd.getNumPartitions()
umbral_local     = MIN_SUPPORT_ABS / num_particiones  # umbral por partición

print(f"Soporte mínimo absoluto: {MIN_SUPPORT_ABS:,} usuarios")
print(f"Particiones:             {num_particiones}")
print(f"Umbral local por partición: {umbral_local:.1f}")

# ── PASADA 1: itemsets frecuentes locales por partición ───────────────────────
def apriori_local(particion, umbral):
    """Apriori simplificado: encuentra 1-itemsets y 2-itemsets frecuentes."""
    transacciones_loc = [list(row["items"]) for row in particion]
    if not transacciones_loc:
        return iter([])

    # Conteo de 1-itemsets
    conteo_1 = {}
    for t in transacciones_loc:
        for item in set(t):
            conteo_1[item] = conteo_1.get(item, 0) + 1

    frecuentes_1 = [item for item, cnt in conteo_1.items() if cnt >= umbral]

    # Conteo de 2-itemsets
    conteo_2 = {}
    freq_set = set(frecuentes_1)
    for t in transacciones_loc:
        items_freq = sorted(set(t) & freq_set)
        for par in itertools.combinations(items_freq, 2):
            conteo_2[par] = conteo_2.get(par, 0) + 1

    frecuentes_2 = [par for par, cnt in conteo_2.items() if cnt >= umbral]

    # Retornar candidatos como frozensets
    candidatos = [frozenset([i]) for i in frecuentes_1]
    candidatos += [frozenset(p) for p in frecuentes_2]
    return iter(candidatos)

t0 = time.time()

candidatos_rdd = (transacciones.rdd
    .mapPartitions(lambda p: apriori_local(p, umbral_local))
    .distinct()
)
candidatos_rdd.cache()
n_candidatos = candidatos_rdd.count()
print(f"\nPasada 1 → Candidatos globales: {n_candidatos:,}")

# ── PASADA 2: contar soporte global de candidatos ─────────────────────────────
candidatos_bc = spark.sparkContext.broadcast(candidatos_rdd.collect())

def contar_soporte(particion):
    candidatos = candidatos_bc.value
    transacciones_loc = [set(row["items"]) for row in particion]
    conteos = {}
    for c in candidatos:
        cnt = sum(1 for t in transacciones_loc if c.issubset(t))
        if cnt > 0:
            conteos[c] = cnt
    return iter(conteos.items())

soporte_global = (transacciones.rdd
    .mapPartitions(contar_soporte)
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda x: x[1] >= MIN_SUPPORT_ABS)
)
soporte_global.cache()

t_son = time.time() - t0
itemsets_son = soporte_global.collect()
print(f"Pasada 2 → Itemsets frecuentes globales: {len(itemsets_son):,}")
print(f"SON completado en {t_son:.1f}s")

# Mostrar top 10
itemsets_son_sorted = sorted(itemsets_son, key=lambda x: x[1], reverse=True)
print("\nTop 10 itemsets frecuentes (SON):")
print(f"{'Itemset':>30} | {'Soporte':>10}")
print("-" * 45)
for itemset, soporte in itemsets_son_sorted[:10]:
    print(f"{str(set(itemset)):>30} | {soporte:>10,}")

# Generar reglas de SON y comparar

In [ ]:
# ── Reglas de asociación desde SON ───────────────────────────────────────────
soporte_dict = {itemset: cnt for itemset, cnt in itemsets_son}

reglas_son = []
for itemset, soporte in itemsets_son:
    if len(itemset) < 2:
        continue
    for item in itemset:
        antecedente = itemset - frozenset([item])
        consecuente = frozenset([item])
        if antecedente in soporte_dict:
            confianza = soporte / soporte_dict[antecedente]
            lift      = confianza / (soporte_dict.get(consecuente, 1) / n_usuarios)
            if confianza >= MIN_CONFIDENCE:
                reglas_son.append({
                    "antecedente": set(antecedente),
                    "consecuente": set(consecuente),
                    "soporte":     soporte / n_usuarios,
                    "confianza":   confianza,
                    "lift":        lift
                })

reglas_son_sorted = sorted(reglas_son, key=lambda x: x["confianza"], reverse=True)
print(f"Reglas SON (confianza ≥ {MIN_CONFIDENCE}): {len(reglas_son_sorted):,}")
print(f"\n{'Antecedente':>15} → {'Consecuente':>12} | {'Soporte':>8} | {'Confianza':>10} | {'Lift':>6}")
print("-" * 65)
for r in reglas_son_sorted[:10]:
    print(f"{str(r['antecedente']):>15} → {str(r['consecuente']):>12} | "
          f"{r['soporte']:>8.4f} | {r['confianza']:>10.4f} | {r['lift']:>6.2f}")

# ── Comparación FP-Growth vs SON ──────────────────────────────────────────────
print("\n" + "="*50)
print("COMPARACIÓN FP-Growth vs SON")
print("="*50)
print(f"{'Métrica':<30} {'FP-Growth':>12} {'SON':>12}")
print("-"*55)
print(f"{'Tiempo (s)':<30} {t_fp:>12.1f} {t_son:>12.1f}")
print(f"{'Itemsets frecuentes':<30} {itemsets_fp.count():>12,} {len(itemsets_son):>12,}")
print(f"{'Reglas generadas':<30} {reglas_fp.count():>12,} {len(reglas_son_sorted):>12,}")
print(f"{'minSupport':<30} {MIN_SUPPORT:>12.2f} {MIN_SUPPORT:>12.2f}")
print(f"{'minConfidence':<30} {MIN_CONFIDENCE:>12.2f} {MIN_CONFIDENCE:>12.2f}")

NameError: name 'itemsets_son' is not defined

# Gráfica comparación y top reglas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Preparar datos de reglas FP-Growth
reglas_fp_pd = reglas_fp.orderBy("confidence", ascending=False).limit(15).toPandas()

fig = plt.figure(figsize=(14, 6))
gs  = gridspec.GridSpec(1, 2)

# Izquierda: top reglas FP-Growth por confianza
ax1 = fig.add_subplot(gs[0])
etiquetas = [f"{str(row['antecedent'])} → {str(row['consequent'])}"
             for _, row in reglas_fp_pd.iterrows()]
etiquetas = [e[:40] + "..." if len(e) > 40 else e for e in etiquetas]
ax1.barh(range(len(reglas_fp_pd)), reglas_fp_pd["confidence"],
         color="steelblue", edgecolor="white")
ax1.set_yticks(range(len(reglas_fp_pd)))
ax1.set_yticklabels(etiquetas, fontsize=7)
ax1.set_xlabel("Confianza", fontsize=11)
ax1.set_title("Top reglas FP-Growth\n(por confianza)", fontsize=11)
ax1.grid(True, alpha=0.3, axis='x')
ax1.invert_yaxis()

# Derecha: scatter soporte vs confianza
ax2 = fig.add_subplot(gs[1])
ax2.scatter(reglas_fp_pd["support"], reglas_fp_pd["confidence"],
            c=reglas_fp_pd["lift"], cmap="viridis", s=80, edgecolors='gray')
sm = plt.cm.ScalarMappable(cmap="viridis")
sm.set_array(reglas_fp_pd["lift"])
plt.colorbar(sm, ax=ax2, label="Lift")
ax2.set_xlabel("Soporte", fontsize=11)
ax2.set_ylabel("Confianza", fontsize=11)
ax2.set_title("Soporte vs Confianza\n(color = Lift)", fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle("Fase 4 — Reglas de Asociación (FP-Growth)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/14_reglas_asociacion.png", dpi=150, bbox_inches='tight')
plt.show()